In [1]:


from types import SimpleNamespace
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [2]:
"""Empirical Tail Index estimation using the Hill estimator."""

from Models.activation import ZiLU, ZiLU_Approx
pre_activations = {}

def make_hook(name, storage):
    def hook(module, input, output):
        storage[name] = input[0].detach().cpu().flatten()
    return hook

import numpy as np 
def hill_estimator(data, k=100):
    """
    k: number of upper order statistics to use.
    lower result -> heavier tail.
    """
    sorted_data = np.sort(np.abs(data))[::-1]  # Sort in descending order
    log_ratios = np.log(sorted_data[:k] / sorted_data[k])
    alpha_hat = k / np.sum(log_ratios)
    return alpha_hat 

for name, acts in pre_activations.items():
    tail_index = hill_estimator(acts.numpy(), k=100)
    print(f"Tail index for {name}: {tail_index}")
    



## 1. ResNet

In [44]:
# Dataset 
from dataset import CIFAR100 
    
resnet_args = SimpleNamespace(
    activation='zilu',
    sigma=1.0,
    inplace=True,
    model='resnet20',
    augment=True,
    resize=False,
    noise=0.0,
    num_workers=12,
    persistent_workers=False,
    prefetch_factor=None,
    pin_memory=True,
    dataset='cifar100',
    num_classes=100,
    data_path='/mnt/research/j.farias/mkang2/Datasets',
    batch_size=128,
    device='cuda',
)

cifar100_resnet_dataset = CIFAR100(resnet_args)

Files already downloaded and verified
Files already downloaded and verified


In [59]:

class ResBlock(nn.Module):
    def __init__(self, 
                 args, 
                 in_channels, 
                 out_channels, 
                 stride=1
                 ):

        super(ResBlock, self).__init__()
        self.args = args
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride 

        # Activation Selection
        self.activation = args.activation

        # Activation function mapping
        self.activation_map = {
            "relu": lambda: nn.ReLU(inplace=args.inplace), 
            "silu": lambda: nn.SiLU(inplace=args.inplace), 
            "gelu": lambda: nn.GELU(), 
            "sigmoid": lambda: nn.Sigmoid(), 

            # Previous Activation Generation
            "gelu_s": lambda: GELU_s(sigma=args.sigma, inplace=args.inplace), 
            "silu_s": lambda: SiLU_s(sigma=args.sigma, inplace=args.inplace), 
            "zilu_old": lambda: ZiLU_Old(sigma=args.sigma, inplace=args.inplace), 

            # Current Activation Generation 
            "arctan": lambda: ArcTan(sigma=args.sigma), 
            "arctan_approx": lambda: ArcTan_Approx(sigma=args.sigma), 
            "zilu": lambda: ZiLU(sigma=args.sigma), 
            "zilu_approx": lambda: ZiLU_Approx(sigma=args.sigma), 
            "squareplus": lambda: SquarePlus(beta=4),

            # Other Activations
            "leaky_relu": lambda: nn.LeakyReLU(inplace=args.inplace), 
            "prelu": lambda: nn.PReLU(), 
            "elu": lambda: nn.ELU(inplace=args.inplace), 
            "hardshrink": lambda: nn.Hardshrink(), 
            "softshrink": lambda: nn.Softshrink(), 
            "tanhshrink": lambda: nn.Tanhshrink(), 
            "softplus": lambda: nn.Softplus(),
            "softsign": lambda: nn.Softsign(), 
            "tanh": lambda: nn.Tanh(),
            "celu": lambda: nn.CELU(inplace=args.inplace),
            "mish": lambda: nn.Mish(inplace=args.inplace), 
            "hardswish": lambda: nn.Hardswish(inplace=args.inplace), 
            "hardsigmoid": lambda: nn.Hardsigmoid(inplace=args.inplace),
            "selu": lambda: nn.SELU(inplace=args.inplace),
            "hardtanh": lambda: nn.Hardtanh(inplace=args.inplace), 
            "identity": lambda: nn.Identity()     
        }

        if self.activation not in self.activation_map:
            raise ValueError(f"Unsupported activation function: {self.activation}")

        self.activation1 = self.activation_map[self.activation]()
        self.final_activation = self.activation_map[self.activation]() 


        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            self.activation1
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        
        # Identity mapping
        if stride != 1 or in_channels != out_channels: 
            self.identity = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False), 
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.identity = nn.Identity()


    def forward(self, x):
        identity = self.identity(x)

        out = self.layer1(x)
        out = self.layer2(out)

        out += identity
        out = self.final_activation(out)

        return out
    
class ResNet_CIFAR(nn.Module):
    def __init__(self, args):
        super(ResNet_CIFAR, self).__init__()
        self.args = args 
        self.num_classes = args.num_classes
        """
        ResNet-20 Params: 0.278M
        ResNet-32 Params: 0.466M
        ResNet-44 Params: 0.661M
        ResNet-56 Params: 0.855M
        """
        
        self.name = f"{args.model} - {args.activation}"

        # Activation Selection
        self.activation = args.activation

        # Activation function mapping
        self.activation_map = {
            "relu": lambda: nn.ReLU(inplace=args.inplace), 
            "silu": lambda: nn.SiLU(inplace=args.inplace), 
            "gelu": lambda: nn.GELU(), 
            "sigmoid": lambda: nn.Sigmoid(), 

            # Previous Activation Generation
            "gelu_s": lambda: GELU_s(sigma=args.sigma, inplace=args.inplace), 
            "silu_s": lambda: SiLU_s(sigma=args.sigma, inplace=args.inplace), 
            "zilu_old": lambda: ZiLU_Old(sigma=args.sigma, inplace=args.inplace), 

            # Current Activation Generation 
            "arctan": lambda: ArcTan(sigma=args.sigma), 
            "arctan_approx": lambda: ArcTan_Approx(sigma=args.sigma), 
            "zilu": lambda: ZiLU(sigma=args.sigma), 
            "zilu_approx": lambda: ZiLU_Approx(sigma=args.sigma), 
            "squareplus": lambda: SquarePlus(beta=4),

            # Other Activations
            "leaky_relu": lambda: nn.LeakyReLU(inplace=args.inplace), 
            "prelu": lambda: nn.PReLU(), 
            "elu": lambda: nn.ELU(inplace=args.inplace), 
            "hardshrink": lambda: nn.Hardshrink(), 
            "softshrink": lambda: nn.Softshrink(), 
            "tanhshrink": lambda: nn.Tanhshrink(), 
            "softplus": lambda: nn.Softplus(),
            "softsign": lambda: nn.Softsign(), 
            "tanh": lambda: nn.Tanh(),
            "celu": lambda: nn.CELU(inplace=args.inplace),
            "mish": lambda: nn.Mish(inplace=args.inplace), 
            "hardswish": lambda: nn.Hardswish(inplace=args.inplace), 
            "hardsigmoid": lambda: nn.Hardsigmoid(inplace=args.inplace),
            "selu": lambda: nn.SELU(inplace=args.inplace), 
            "hardtanh": lambda: nn.Hardtanh(inplace=args.inplace), 
            "identity": lambda: nn.Identity()
        }
        
        if self.activation not in self.activation_map:
            raise ValueError(f"Unsupported activation function: {self.activation}")
            
        self.activation_first_conv = self.activation_map[self.activation]()

        self.first_conv = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16), 
            self.activation_first_conv
            )

        if args.model == 'resnet20':
            n = 3 
        elif args.model == 'resnet32':
            n = 5
        elif args.model == 'resnet44':
            n = 7
        elif args.model == 'resnet56':
            n = 9
        else: 
            raise ValueError("Invalid ResNet model type")

        self.in_channels = 16 
        block = ResBlock 
        self.layer1 = self._make_layer(block, 16, n, stride=1)
        self.layer2 = self._make_layer(block, 32, n, stride=2)
        self.layer3 = self._make_layer(block, 64, n, stride=2)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(start_dim=1),
            nn.Linear(64, self.num_classes)
        )

    def _make_layer(self, block, out_channels, blocks, stride=1):
        layers = [] 

        # First block handles stride and channel chang
        layers.append(block(self.args, self.in_channels, out_channels, stride=stride))
        self.in_channels = out_channels

        for _ in range(1, blocks):
            layers.append(block(self.args, self.in_channels, out_channels, stride=1))

        return nn.Sequential(*layers)

    def parameter_count(self): 
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total_params, trainable_params

    def forward(self, x):
        x = self.first_conv(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.classifier(x)

        return x

resnet_batchnorm = ResNet_CIFAR(resnet_args).to(resnet_args.device) 

In [60]:
resnet_batchnorm_pre_activations = {}
for name, module in resnet_batchnorm.named_modules():
    if isinstance(module, ZiLU):
        module.register_forward_hook(make_hook(name, resnet_batchnorm_pre_activations))

# resnet_batchnorm.eval()
for images, labels in cifar100_resnet_dataset.test_loader:
    images, labels = images.to(resnet_args.device), labels.to(resnet_args.device)
    outputs = resnet_batchnorm(images)
    break

hill_estimations = 0
for name, acts in resnet_batchnorm_pre_activations.items():
    tail_index = hill_estimator(acts.numpy(), k=100)
    print(f"Tail index for {name}: {tail_index}")
    hill_estimations += tail_index 

print()
print(f"Average Tail Index: {hill_estimations/len(resnet_batchnorm_pre_activations)}")

    


Tail index for activation_first_conv: 12.879852886751124
Tail index for layer1.0.activation1: 10.139959986566131
Tail index for layer1.0.final_activation: 10.56228374573787
Tail index for layer1.1.activation1: 11.824937747371102
Tail index for layer1.1.final_activation: 9.984705511100836
Tail index for layer1.2.activation1: 8.644110621757921
Tail index for layer1.2.final_activation: 7.433805290812638
Tail index for layer2.0.activation1: 9.501843187726593
Tail index for layer2.0.final_activation: 9.169578828089461
Tail index for layer2.1.activation1: 8.99174782878106
Tail index for layer2.1.final_activation: 6.768531764963778
Tail index for layer2.2.activation1: 10.274963976539308
Tail index for layer2.2.final_activation: 6.502664224711322
Tail index for layer3.0.activation1: 6.264147823412702
Tail index for layer3.0.final_activation: 5.808336677200865
Tail index for layer3.1.activation1: 6.988821428859454
Tail index for layer3.1.final_activation: 6.377643603206606
Tail index for layer3

In [63]:

class ResBlock(nn.Module):
    def __init__(self, 
                 args, 
                 in_channels, 
                 out_channels, 
                 stride=1
                 ):

        super(ResBlock, self).__init__()
        self.args = args
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride 

        # Activation Selection
        self.activation = args.activation

        # Activation function mapping
        self.activation_map = {
            "relu": lambda: nn.ReLU(inplace=args.inplace), 
            "silu": lambda: nn.SiLU(inplace=args.inplace), 
            "gelu": lambda: nn.GELU(), 
            "sigmoid": lambda: nn.Sigmoid(), 

            # Previous Activation Generation
            "gelu_s": lambda: GELU_s(sigma=args.sigma, inplace=args.inplace), 
            "silu_s": lambda: SiLU_s(sigma=args.sigma, inplace=args.inplace), 
            "zilu_old": lambda: ZiLU_Old(sigma=args.sigma, inplace=args.inplace), 

            # Current Activation Generation 
            "arctan": lambda: ArcTan(sigma=args.sigma), 
            "arctan_approx": lambda: ArcTan_Approx(sigma=args.sigma), 
            "zilu": lambda: ZiLU(sigma=args.sigma), 
            "zilu_approx": lambda: ZiLU_Approx(sigma=args.sigma), 
            "squareplus": lambda: SquarePlus(beta=4),

            # Other Activations
            "leaky_relu": lambda: nn.LeakyReLU(inplace=args.inplace), 
            "prelu": lambda: nn.PReLU(), 
            "elu": lambda: nn.ELU(inplace=args.inplace), 
            "hardshrink": lambda: nn.Hardshrink(), 
            "softshrink": lambda: nn.Softshrink(), 
            "tanhshrink": lambda: nn.Tanhshrink(), 
            "softplus": lambda: nn.Softplus(),
            "softsign": lambda: nn.Softsign(), 
            "tanh": lambda: nn.Tanh(),
            "celu": lambda: nn.CELU(inplace=args.inplace),
            "mish": lambda: nn.Mish(inplace=args.inplace), 
            "hardswish": lambda: nn.Hardswish(inplace=args.inplace), 
            "hardsigmoid": lambda: nn.Hardsigmoid(inplace=args.inplace),
            "selu": lambda: nn.SELU(inplace=args.inplace),
            "hardtanh": lambda: nn.Hardtanh(inplace=args.inplace), 
            "identity": lambda: nn.Identity()     
        }

        if self.activation not in self.activation_map:
            raise ValueError(f"Unsupported activation function: {self.activation}")

        self.activation1 = self.activation_map[self.activation]()
        self.final_activation = self.activation_map[self.activation]() 


        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.Identity(),
            self.activation1
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.Identity(),
        )
        
        # Identity mapping
        if stride != 1 or in_channels != out_channels: 
            self.identity = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False), 
                nn.Identity()
            )
        else:
            self.identity = nn.Identity()


    def forward(self, x):
        identity = self.identity(x)

        out = self.layer1(x)
        out = self.layer2(out)

        out += identity
        out = self.final_activation(out)

        return out

    
class ResNet_CIFAR(nn.Module):
    def __init__(self, args):
        super(ResNet_CIFAR, self).__init__()
        self.args = args 
        self.num_classes = args.num_classes
        """
        ResNet-20 Params: 0.278M
        ResNet-32 Params: 0.466M
        ResNet-44 Params: 0.661M
        ResNet-56 Params: 0.855M
        """
        
        self.name = f"{args.model} - {args.activation}"

        # Activation Selection
        self.activation = args.activation

        # Activation function mapping
        self.activation_map = {
            "relu": lambda: nn.ReLU(inplace=args.inplace), 
            "silu": lambda: nn.SiLU(inplace=args.inplace), 
            "gelu": lambda: nn.GELU(), 
            "sigmoid": lambda: nn.Sigmoid(), 

            # Previous Activation Generation
            "gelu_s": lambda: GELU_s(sigma=args.sigma, inplace=args.inplace), 
            "silu_s": lambda: SiLU_s(sigma=args.sigma, inplace=args.inplace), 
            "zilu_old": lambda: ZiLU_Old(sigma=args.sigma, inplace=args.inplace), 

            # Current Activation Generation 
            "arctan": lambda: ArcTan(sigma=args.sigma), 
            "arctan_approx": lambda: ArcTan_Approx(sigma=args.sigma), 
            "zilu": lambda: ZiLU(sigma=args.sigma), 
            "zilu_approx": lambda: ZiLU_Approx(sigma=args.sigma), 
            "squareplus": lambda: SquarePlus(beta=4),

            # Other Activations
            "leaky_relu": lambda: nn.LeakyReLU(inplace=args.inplace), 
            "prelu": lambda: nn.PReLU(), 
            "elu": lambda: nn.ELU(inplace=args.inplace), 
            "hardshrink": lambda: nn.Hardshrink(), 
            "softshrink": lambda: nn.Softshrink(), 
            "tanhshrink": lambda: nn.Tanhshrink(), 
            "softplus": lambda: nn.Softplus(),
            "softsign": lambda: nn.Softsign(), 
            "tanh": lambda: nn.Tanh(),
            "celu": lambda: nn.CELU(inplace=args.inplace),
            "mish": lambda: nn.Mish(inplace=args.inplace), 
            "hardswish": lambda: nn.Hardswish(inplace=args.inplace), 
            "hardsigmoid": lambda: nn.Hardsigmoid(inplace=args.inplace),
            "selu": lambda: nn.SELU(inplace=args.inplace), 
            "hardtanh": lambda: nn.Hardtanh(inplace=args.inplace), 
            "identity": lambda: nn.Identity()
        }
        
        if self.activation not in self.activation_map:
            raise ValueError(f"Unsupported activation function: {self.activation}")
            
        self.activation_first_conv = self.activation_map[self.activation]()

        self.first_conv = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.Identity(), 
            self.activation_first_conv
            )

        if args.model == 'resnet20':
            n = 3 
        elif args.model == 'resnet32':
            n = 5
        elif args.model == 'resnet44':
            n = 7
        elif args.model == 'resnet56':
            n = 9
        else: 
            raise ValueError("Invalid ResNet model type")

        self.in_channels = 16 
        block = ResBlock 
        self.layer1 = self._make_layer(block, 16, n, stride=1)
        self.layer2 = self._make_layer(block, 32, n, stride=2)
        self.layer3 = self._make_layer(block, 64, n, stride=2)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(start_dim=1),
            nn.Linear(64, self.num_classes)
        )

    def _make_layer(self, block, out_channels, blocks, stride=1):
        layers = [] 

        # First block handles stride and channel chang
        layers.append(block(self.args, self.in_channels, out_channels, stride=stride))
        self.in_channels = out_channels

        for _ in range(1, blocks):
            layers.append(block(self.args, self.in_channels, out_channels, stride=1))

        return nn.Sequential(*layers)

    def parameter_count(self): 
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total_params, trainable_params

    def forward(self, x):
        x = self.first_conv(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.classifier(x)

        return x

resnet_nobatchnorm = ResNet_CIFAR(resnet_args).to(resnet_args.device)

In [65]:

resnet_nobatchnorm_pre_activations = {}
for name, module in resnet_nobatchnorm.named_modules():
    if isinstance(module, ZiLU):
        module.register_forward_hook(make_hook(name, resnet_nobatchnorm_pre_activations))

# resnet_nobatchnorm.eval()
for images, labels in cifar100_resnet_dataset.test_loader:
    images, labels = images.to(resnet_args.device), labels.to(resnet_args.device)
    outputs = resnet_nobatchnorm(images)
    break

hill_estimations = 0
for name, acts in resnet_nobatchnorm_pre_activations.items():
    tail_index = hill_estimator(acts.numpy(), k=100)
    print(f"Tail index for {name}: {tail_index}")
    hill_estimations += tail_index 

print()
print(f"Average Tail Index: {hill_estimations/len(resnet_nobatchnorm_pre_activations)}")

    


Tail index for activation_first_conv: 581.0917627051574
Tail index for layer1.0.activation1: 24188.062213408703
Tail index for layer1.0.final_activation: 43.866502801490014
Tail index for layer1.1.activation1: 23.8434654084475
Tail index for layer1.1.final_activation: 39.10770772529172
Tail index for layer1.2.activation1: 62.20947008981824
Tail index for layer1.2.final_activation: 29.025720640698072
Tail index for layer2.0.activation1: 10.31245912441422
Tail index for layer2.0.final_activation: 23.653185741535097
Tail index for layer2.1.activation1: 14.262545522752417
Tail index for layer2.1.final_activation: 15.161054832194397
Tail index for layer2.2.activation1: 8.068323940079127
Tail index for layer2.2.final_activation: 11.553605449777303
Tail index for layer3.0.activation1: 9.751887553308315
Tail index for layer3.0.final_activation: 8.12283092172072
Tail index for layer3.1.activation1: 7.076131155846954
Tail index for layer3.1.final_activation: 9.049786877215245
Tail index for laye

## II. ViT

In [66]:
# Dataset 
from dataset import CIFAR100 

    
vit_args = SimpleNamespace(
    activation='zilu',
    sigma=1.0,
    inplace=True,
    model='vit-tiny',
    augment=True,
    resize=224,
    img_size=(3, 224, 224),
    patch_size=16,
    d_hidden=192,
    d_mlp=768,
    num_heads=3,
    num_layers=12,
    dropout=0.1,
    attention_dropout=0.1,
    drop_path_rate=0.0,
    noise=0.0,
    num_workers=12,
    persistent_workers=False,
    prefetch_factor=None,
    pin_memory=True,
    dataset='cifar100',
    num_classes=100,    
    data_path='/mnt/research/j.farias/mkang2/Datasets',
    batch_size=128,
    device='cuda',
)

cifar100_vit_dataset = CIFAR100(vit_args)

Files already downloaded and verified
Files already downloaded and verified


In [67]:
class ViT(nn.Module): 
    def __init__(self, args): 
        super(ViT, self).__init__()
        assert args.img_size[1] % args.patch_size == 0 and args.img_size[2] % args.patch_size == 0, "img_size dimensions must be divisible by patch_size dimensions"
        assert args.d_hidden % args.num_heads == 0, "d_hidden must be divisible by n_heads"
        
        self.args = args
        self.model = "VIT"
        
        self.d_hidden = self.args.d_hidden 
        self.d_mlp = self.args.d_mlp
        
        self.img_size = self.args.img_size[1:]
        self.n_classes = self.args.num_classes # Number of Classes
        self.n_heads = self.args.num_heads
        self.patch_size = (self.args.patch_size, self.args.patch_size) # Patch Size
        self.n_channels = self.args.img_size[0]
        self.n_layers = self.args.num_layers # Number of Layers
        
        self.n_patches = (self.img_size[0] * self.img_size[1]) // (self.patch_size[0] * self.patch_size[1])
        
        self.dropout = self.args.dropout # Dropout Rate
        self.attention_dropout = self.args.attention_dropout # Attention Dropout Rate   
        self.max_seq_length = self.n_patches + 1 # +1 for class token
        
        self.patch_embedding = PatchEmbedding(self.d_hidden, self.img_size, self.patch_size, self.n_channels) # Patch Embedding Layer
        self.positional_encoding = PositionalEncoding(self.d_hidden, self.max_seq_length)

        self.dpr = [x.item() for x in torch.linspace(0, self.args.drop_path_rate, self.n_layers)]  # stochastic depth decay rule

        # self.transformer_encoder = nn.Sequential(*[TransformerEncoder_DropPath(
        #     args=args, 
        #     d_hidden=self.d_hidden, 
        #     d_mlp=self.d_mlp, 
        #     num_heads=self.n_heads, 
        #     dropout=self.dropout, 
        #     attention_dropout=self.attention_dropout,
        #     drop_path=self.dpr[i]
        #     ) for i in range(self.n_layers)])
        
        self.transformer_encoder = nn.Sequential(*[TransformerEncoder(
            args=args, 
            d_hidden=self.d_hidden, 
            d_mlp=self.d_mlp, 
            num_heads=self.n_heads, 
            dropout=self.dropout, 
            attention_dropout=self.attention_dropout
            ) for _ in range(self.n_layers)])
        
        self.classifier = nn.Linear(self.d_hidden, self.n_classes)
        
        self.device = args.device
        
        self.to(self.device)
        self.name = f"{self.args.model} {self.args.activation}"
        
    def forward(self, x): 
        x = self.patch_embedding(x)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        x = self.classifier(x[:, 0]) # Taking the CLS token for classification
        return x

    def summary(self): 
        original_device = next(self.parameters()).device
        try:
            self.to("cpu")
            print(f"--- Summary for {self.name} ---")
            summary(self, input_size=self.img_size, device="cpu") 
        except Exception as e:
            print(f"Could not generate summary: {e}")
        finally:
            self.to(original_device)
        
    def parameter_count(self): 
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total_params, trainable_params
        
class PatchEmbedding(nn.Module): 
    def __init__(self, d_hidden, img_size, patch_size, n_channels=3): 
        super(PatchEmbedding, self).__init__()
        
        self.d_hidden = d_hidden # Dimensionality of Model 
        self.img_size = img_size # Size of Image
        self.patch_size = patch_size # Patch Size 
        self.n_channels = n_channels # Number of Channels in Image
        
        self.linear_projection = nn.Conv2d(in_channels=n_channels, out_channels=d_hidden, kernel_size=patch_size, stride=patch_size) # Linear Projection Layer
        self.norm = nn.LayerNorm(d_hidden) # Normalization Layer
        
        self.flatten = nn.Flatten(start_dim=2)
        
    def forward(self, x): 
        x = self.linear_projection(x) # (B, C, H, W) -> (B, d_hidden, H', W')
        x = self.flatten(x) # (B, d_hidden, H', W') -> (B, d_hidden, n_patches)
        x = x.transpose(1, 2) # (B, d_hidden, n_patches) -> (B, n_patches, d_hidden)
        x = self.norm(x) # (B, n_patches, d_hidden) -> (B, n_patches, d_hidden)
        return x
    
class PositionalEncoding(nn.Module):
    def __init__(self, d_hidden, max_seq_length): 
        super(PositionalEncoding, self).__init__()
        
        self.cls_tokens = nn.Parameter(torch.randn(1, 1, d_hidden)) # Classification Token

        pe = torch.zeros(max_seq_length, d_hidden)  # Positional Encoding Tensor
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1) 
        div_term = torch.exp(torch.arange(0, d_hidden, 2).float() * (-np.log(10000.0) / d_hidden))  

        pe[:, 0::2] = torch.sin(position * div_term)  # Apply sine to even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # Apply cosine to odd indices

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x): 
        # Expand to have class token for each image in batch 
        tokens_batch = self.cls_tokens.expand(x.shape[0], -1, -1) # (B, 1, d_hidden)
        
        # Concatenate class token with positional encoding
        x = torch.cat((tokens_batch, x), dim=1)
        
        # Add positional encoding to the input 
        x = x + self.pe[:, :x.size(1)].to(x.device) 
        return x

"""Multi-Head Layers for Transformer Encoder"""
class MultiHeadAttention(nn.Module): 
    def __init__(self, d_hidden, num_heads, attention_dropout):
        super(MultiHeadAttention, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"
        
        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.d_k = d_hidden // num_heads # dimension of each head
        self.dropout = nn.Dropout(attention_dropout)
        
        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden) # Final Transformation before residual connection
    
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        attn_probs = self.dropout(torch.softmax(attn_scores, dim=-1))
        output = torch.matmul(attn_probs, V)
        return output, attn_probs
    
    def split_head(self, x): 
        batch_size, seq_length, d_hidden = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) # (B, num_heads, seq_length, d_k)
        
    def combine_heads(self, x): 
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden) 
    
    def forward(self, x, mask=None):
        q = self.split_head(self.W_q(x)) # (B, num_heads, seq_length, d_k)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))
        
        attn_output, _ = self.scaled_dot_product_attention(q, k, v, mask) # (B, num_heads, seq_length, d_k)
        output = self.W_o(self.combine_heads(attn_output)) # (B, seq_length, d_hidden)
        return output


class TransformerEncoder(nn.Module): 
    def __init__(self, args, d_hidden, d_mlp, num_heads, dropout, attention_dropout):
        super(TransformerEncoder, self).__init__()
        self.args = args 

        self.d_hidden = d_hidden 
        self.d_mlp = d_mlp
        self.num_heads = num_heads
        self.dropout = dropout
        self.attention_dropout = attention_dropout

        self.attention = MultiHeadAttention(d_hidden, num_heads, attention_dropout)

        self.norm1 = nn.LayerNorm(d_hidden)
        self.norm2 = nn.LayerNorm(d_hidden)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        # Activation Selection
        self.activation = args.activation

        # Activation function mapping
        self.activation_map = {
            "relu": lambda: nn.ReLU(inplace=args.inplace), 
            "silu": lambda: nn.SiLU(inplace=args.inplace), 
            "gelu": lambda: nn.GELU(), 
            "sigmoid": lambda: nn.Sigmoid(), 

            # Previous Activation Generation
            "gelu_s": lambda: GELU_s(sigma=args.sigma, inplace=args.inplace), 
            "silu_s": lambda: SiLU_s(sigma=args.sigma, inplace=args.inplace), 
            "zilu_old": lambda: ZiLU_Old(sigma=args.sigma, inplace=args.inplace), 

            # Current Activation Generation 
            "arctan": lambda: ArcTan(sigma=args.sigma), 
            "arctan_approx": lambda: ArcTan_Approx(sigma=args.sigma), 
            "zilu": lambda: ZiLU(sigma=args.sigma), 
            "zilu_approx": lambda: ZiLU_Approx(sigma=args.sigma), 
            "squareplus": lambda: SquarePlus(beta=4),

            # Other Activations
            "leaky_relu": lambda: nn.LeakyReLU(inplace=args.inplace), 
            "prelu": lambda: nn.PReLU(), 
            "elu": lambda: nn.ELU(inplace=args.inplace), 
            "hardshrink": lambda: nn.Hardshrink(), 
            "softshrink": lambda: nn.Softshrink(), 
            "tanhshrink": lambda: nn.Tanhshrink(), 
            "softplus": lambda: nn.Softplus(),
            "softsign": lambda: nn.Softsign(), 
            "tanh": lambda: nn.Tanh(),
            "celu": lambda: nn.CELU(inplace=args.inplace),
            "mish": lambda: nn.Mish(inplace=args.inplace), 
            "hardswish": lambda: nn.Hardswish(inplace=args.inplace), 
            "hardsigmoid": lambda: nn.Hardsigmoid(inplace=args.inplace),
            "selu": lambda: nn.SELU(inplace=args.inplace),
            "hardtanh": lambda: nn.Hardtanh(inplace=args.inplace), 
            "identity": lambda: nn.Identity()
        }

        if self.activation not in self.activation_map:
            raise ValueError(f"Unsupported activation function: {self.activation}")
            

        # Multilayer Perceptron 
        self.mlp = nn.Sequential(
            nn.Linear(d_hidden, d_mlp),
            self.activation_map[self.activation](), 
            nn.Dropout(dropout),
            nn.Linear(d_mlp, d_hidden)
        )
        
    def forward(self, x): 
        # Pre-Norm Multi-Head Attention 
        norm_x = self.norm1(x) 
        attn_output = self.attention(norm_x)  
        x = x + self.dropout1(attn_output)
        
        # Post-Norm Feed Forward Network
        norm_x = self.norm2(x)  
        mlp_output = self.mlp(norm_x)
        x = x + self.dropout2(mlp_output)  
        return x

"""Transformer Encoder with DropPath - Stochastic Depth"""
# Used for Swin Transformer Experiments
class TransformerEncoder_DropPath(nn.Module): 
    def __init__(self, args, d_hidden, d_mlp, num_heads, dropout, attention_dropout, drop_path):
        super(TransformerEncoder_DropPath, self).__init__()
        self.args = args 

        self.d_hidden = d_hidden 
        self.d_mlp = d_mlp
        self.num_heads = num_heads
        self.dropout = dropout
        self.attention_dropout = attention_dropout

        self.attention = MultiHeadAttention(d_hidden, num_heads, attention_dropout)

        self.norm1 = nn.LayerNorm(d_hidden)
        self.norm2 = nn.LayerNorm(d_hidden)
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

        # Activation Selection
        self.activation = args.activation

        # Activation function mapping
        self.activation_map = {
            "relu": lambda: nn.ReLU(inplace=args.inplace), 
            "silu": lambda: nn.SiLU(inplace=args.inplace), 
            "gelu": lambda: nn.GELU(), 
            "sigmoid": lambda: nn.Sigmoid(), 

            # Previous Activation Generation
            "gelu_s": lambda: GELU_s(sigma=args.sigma, inplace=args.inplace), 
            "silu_s": lambda: SiLU_s(sigma=args.sigma, inplace=args.inplace), 
            "zilu_old": lambda: ZiLU_Old(sigma=args.sigma, inplace=args.inplace), 

            # Current Activation Generation 
            "arctan": lambda: ArcTan(sigma=args.sigma), 
            "arctan_approx": lambda: ArcTan_Approx(sigma=args.sigma), 
            "zilu": lambda: ZiLU(sigma=args.sigma), 
            "zilu_approx": lambda: ZiLU_Approx(sigma=args.sigma), 
            "squareplus": lambda: SquarePlus(beta=4),

            # Other Activations
            "leaky_relu": lambda: nn.LeakyReLU(inplace=args.inplace), 
            "prelu": lambda: nn.PReLU(), 
            "elu": lambda: nn.ELU(inplace=args.inplace), 
            "hardshrink": lambda: nn.Hardshrink(), 
            "softshrink": lambda: nn.Softshrink(), 
            "tanhshrink": lambda: nn.Tanhshrink(), 
            "softplus": lambda: nn.Softplus(),
            "softsign": lambda: nn.Softsign(), 
            "tanh": lambda: nn.Tanh(),
            "celu": lambda: nn.CELU(inplace=args.inplace),
            "mish": lambda: nn.Mish(inplace=args.inplace), 
            "hardswish": lambda: nn.Hardswish(inplace=args.inplace), 
            "hardsigmoid": lambda: nn.Hardsigmoid(inplace=args.inplace),
            "selu": lambda: nn.SELU(inplace=args.inplace),
            "hardtanh": lambda: nn.Hardtanh(inplace=args.inplace), 
            "identity": lambda: nn.Identity()
        }

        if self.activation not in self.activation_map:
            raise ValueError(f"Unsupported activation function: {self.activation}")
            

        # Multilayer Perceptron 
        self.mlp = nn.Sequential(
            nn.Linear(d_hidden, d_mlp),
            self.activation_map[self.activation](), 
            nn.Dropout(dropout),
            nn.Linear(d_mlp, d_hidden)
        )
        
    def forward(self, x): 
        # Pre-Norm Multi-Head Attention 
        norm_x = self.norm1(x) 
        attn_output = self.attention(norm_x)  
        x = x + self.drop_path(attn_output)
        
        # Post-Norm Feed Forward Network
        norm_x = self.norm2(x)  
        mlp_output = self.mlp(norm_x)
        x = x + self.drop_path(mlp_output)  
        return x

"""Also can use timm's DropPath Implementation"""
# from timm.models.layers import DropPath
class DropPath(nn.Module):
    """Drop paths (Stochastic Depth) per sample  (when applied in main path of residual blocks)."""
    def __init__(self, drop_prob: float = 0.0):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob
        
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()  # binarize
        output = x.div(keep_prob) * random_tensor
        return output

vit_layernorm = ViT(vit_args).to(vit_args.device)

In [68]:
vit_layernorm_pre_activations = {}

for name, module in vit_layernorm.named_modules():
    if isinstance(module, ZiLU):
        module.register_forward_hook(make_hook(name, vit_layernorm_pre_activations))

# vit_layernorm.eval()
for images, labels in cifar100_vit_dataset.test_loader:
    images, labels = images.to(vit_layernorm.device), labels.to(vit_layernorm.device)
    outputs = vit_layernorm(images)
    break


hill_estimations = 0
for name, acts in vit_layernorm_pre_activations.items():
    tail_index = hill_estimator(acts.numpy(), k=100)
    print(f"Tail index for {name}: {tail_index}")
    hill_estimations += tail_index 

print()
print(f"Average Tail Index: {hill_estimations/len(vit_layernorm_pre_activations)}")



Tail index for transformer_encoder.0.mlp.1: 28.847042388323477
Tail index for transformer_encoder.1.mlp.1: 35.48516867220045
Tail index for transformer_encoder.2.mlp.1: 34.99511363785219
Tail index for transformer_encoder.3.mlp.1: 40.38236263987542
Tail index for transformer_encoder.4.mlp.1: 35.72912674512932
Tail index for transformer_encoder.5.mlp.1: 31.114774782153017
Tail index for transformer_encoder.6.mlp.1: 32.87283352064944
Tail index for transformer_encoder.7.mlp.1: 25.74899970483496
Tail index for transformer_encoder.8.mlp.1: 29.353888243975977
Tail index for transformer_encoder.9.mlp.1: 40.374479364351146
Tail index for transformer_encoder.10.mlp.1: 29.804427064920834
Tail index for transformer_encoder.11.mlp.1: 26.092474757795838

Average Tail Index: 32.566724293505175


In [69]:
class ViT(nn.Module): 
    def __init__(self, args): 
        super(ViT, self).__init__()
        assert args.img_size[1] % args.patch_size == 0 and args.img_size[2] % args.patch_size == 0, "img_size dimensions must be divisible by patch_size dimensions"
        assert args.d_hidden % args.num_heads == 0, "d_hidden must be divisible by n_heads"
        
        self.args = args
        self.model = "VIT"
        
        self.d_hidden = self.args.d_hidden 
        self.d_mlp = self.args.d_mlp
        
        self.img_size = self.args.img_size[1:]
        self.n_classes = self.args.num_classes # Number of Classes
        self.n_heads = self.args.num_heads
        self.patch_size = (self.args.patch_size, self.args.patch_size) # Patch Size
        self.n_channels = self.args.img_size[0]
        self.n_layers = self.args.num_layers # Number of Layers
        
        self.n_patches = (self.img_size[0] * self.img_size[1]) // (self.patch_size[0] * self.patch_size[1])
        
        self.dropout = self.args.dropout # Dropout Rate
        self.attention_dropout = self.args.attention_dropout # Attention Dropout Rate   
        self.max_seq_length = self.n_patches + 1 # +1 for class token
        
        self.patch_embedding = PatchEmbedding(self.d_hidden, self.img_size, self.patch_size, self.n_channels) # Patch Embedding Layer
        self.positional_encoding = PositionalEncoding(self.d_hidden, self.max_seq_length)

        self.dpr = [x.item() for x in torch.linspace(0, self.args.drop_path_rate, self.n_layers)]  # stochastic depth decay rule

        # self.transformer_encoder = nn.Sequential(*[TransformerEncoder_DropPath(
        #     args=args, 
        #     d_hidden=self.d_hidden, 
        #     d_mlp=self.d_mlp, 
        #     num_heads=self.n_heads, 
        #     dropout=self.dropout, 
        #     attention_dropout=self.attention_dropout,
        #     drop_path=self.dpr[i]
        #     ) for i in range(self.n_layers)])
        
        self.transformer_encoder = nn.Sequential(*[TransformerEncoder(
            args=args, 
            d_hidden=self.d_hidden, 
            d_mlp=self.d_mlp, 
            num_heads=self.n_heads, 
            dropout=self.dropout, 
            attention_dropout=self.attention_dropout
            ) for _ in range(self.n_layers)])
        
        self.classifier = nn.Linear(self.d_hidden, self.n_classes)
        
        self.device = args.device
        
        self.to(self.device)
        self.name = f"{self.args.model} {self.args.activation}"
        
    def forward(self, x): 
        x = self.patch_embedding(x)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        x = self.classifier(x[:, 0]) # Taking the CLS token for classification
        return x

    def summary(self): 
        original_device = next(self.parameters()).device
        try:
            self.to("cpu")
            print(f"--- Summary for {self.name} ---")
            summary(self, input_size=self.img_size, device="cpu") 
        except Exception as e:
            print(f"Could not generate summary: {e}")
        finally:
            self.to(original_device)
        
    def parameter_count(self): 
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total_params, trainable_params
        
class PatchEmbedding(nn.Module): 
    def __init__(self, d_hidden, img_size, patch_size, n_channels=3): 
        super(PatchEmbedding, self).__init__()
        
        self.d_hidden = d_hidden # Dimensionality of Model 
        self.img_size = img_size # Size of Image
        self.patch_size = patch_size # Patch Size 
        self.n_channels = n_channels # Number of Channels in Image
        
        self.linear_projection = nn.Conv2d(in_channels=n_channels, out_channels=d_hidden, kernel_size=patch_size, stride=patch_size) # Linear Projection Layer
        self.norm = nn.Identity()
        
        self.flatten = nn.Flatten(start_dim=2)
        
    def forward(self, x): 
        x = self.linear_projection(x) # (B, C, H, W) -> (B, d_hidden, H', W')
        x = self.flatten(x) # (B, d_hidden, H', W') -> (B, d_hidden, n_patches)
        x = x.transpose(1, 2) # (B, d_hidden, n_patches) -> (B, n_patches, d_hidden)
        x = self.norm(x) # (B, n_patches, d_hidden) -> (B, n_patches, d_hidden)
        return x
    
class PositionalEncoding(nn.Module):
    def __init__(self, d_hidden, max_seq_length): 
        super(PositionalEncoding, self).__init__()
        
        self.cls_tokens = nn.Parameter(torch.randn(1, 1, d_hidden)) # Classification Token

        pe = torch.zeros(max_seq_length, d_hidden)  # Positional Encoding Tensor
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1) 
        div_term = torch.exp(torch.arange(0, d_hidden, 2).float() * (-np.log(10000.0) / d_hidden))  

        pe[:, 0::2] = torch.sin(position * div_term)  # Apply sine to even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # Apply cosine to odd indices

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x): 
        # Expand to have class token for each image in batch 
        tokens_batch = self.cls_tokens.expand(x.shape[0], -1, -1) # (B, 1, d_hidden)
        
        # Concatenate class token with positional encoding
        x = torch.cat((tokens_batch, x), dim=1)
        
        # Add positional encoding to the input 
        x = x + self.pe[:, :x.size(1)].to(x.device) 
        return x

"""Multi-Head Layers for Transformer Encoder"""
class MultiHeadAttention(nn.Module): 
    def __init__(self, d_hidden, num_heads, attention_dropout):
        super(MultiHeadAttention, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"
        
        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.d_k = d_hidden // num_heads # dimension of each head
        self.dropout = nn.Dropout(attention_dropout)
        
        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden) # Final Transformation before residual connection
    
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        attn_probs = self.dropout(torch.softmax(attn_scores, dim=-1))
        output = torch.matmul(attn_probs, V)
        return output, attn_probs
    
    def split_head(self, x): 
        batch_size, seq_length, d_hidden = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) # (B, num_heads, seq_length, d_k)
        
    def combine_heads(self, x): 
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden) 
    
    def forward(self, x, mask=None):
        q = self.split_head(self.W_q(x)) # (B, num_heads, seq_length, d_k)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))
        
        attn_output, _ = self.scaled_dot_product_attention(q, k, v, mask) # (B, num_heads, seq_length, d_k)
        output = self.W_o(self.combine_heads(attn_output)) # (B, seq_length, d_hidden)
        return output


class TransformerEncoder(nn.Module): 
    def __init__(self, args, d_hidden, d_mlp, num_heads, dropout, attention_dropout):
        super(TransformerEncoder, self).__init__()
        self.args = args 

        self.d_hidden = d_hidden 
        self.d_mlp = d_mlp
        self.num_heads = num_heads
        self.dropout = dropout
        self.attention_dropout = attention_dropout

        self.attention = MultiHeadAttention(d_hidden, num_heads, attention_dropout)

        self.norm1 = nn.Identity()
        self.norm2 = nn.Identity()
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        # Activation Selection
        self.activation = args.activation

        # Activation function mapping
        self.activation_map = {
            "relu": lambda: nn.ReLU(inplace=args.inplace), 
            "silu": lambda: nn.SiLU(inplace=args.inplace), 
            "gelu": lambda: nn.GELU(), 
            "sigmoid": lambda: nn.Sigmoid(), 

            # Previous Activation Generation
            "gelu_s": lambda: GELU_s(sigma=args.sigma, inplace=args.inplace), 
            "silu_s": lambda: SiLU_s(sigma=args.sigma, inplace=args.inplace), 
            "zilu_old": lambda: ZiLU_Old(sigma=args.sigma, inplace=args.inplace), 

            # Current Activation Generation 
            "arctan": lambda: ArcTan(sigma=args.sigma), 
            "arctan_approx": lambda: ArcTan_Approx(sigma=args.sigma), 
            "zilu": lambda: ZiLU(sigma=args.sigma), 
            "zilu_approx": lambda: ZiLU_Approx(sigma=args.sigma), 
            "squareplus": lambda: SquarePlus(beta=4),

            # Other Activations
            "leaky_relu": lambda: nn.LeakyReLU(inplace=args.inplace), 
            "prelu": lambda: nn.PReLU(), 
            "elu": lambda: nn.ELU(inplace=args.inplace), 
            "hardshrink": lambda: nn.Hardshrink(), 
            "softshrink": lambda: nn.Softshrink(), 
            "tanhshrink": lambda: nn.Tanhshrink(), 
            "softplus": lambda: nn.Softplus(),
            "softsign": lambda: nn.Softsign(), 
            "tanh": lambda: nn.Tanh(),
            "celu": lambda: nn.CELU(inplace=args.inplace),
            "mish": lambda: nn.Mish(inplace=args.inplace), 
            "hardswish": lambda: nn.Hardswish(inplace=args.inplace), 
            "hardsigmoid": lambda: nn.Hardsigmoid(inplace=args.inplace),
            "selu": lambda: nn.SELU(inplace=args.inplace),
            "hardtanh": lambda: nn.Hardtanh(inplace=args.inplace), 
            "identity": lambda: nn.Identity()
        }

        if self.activation not in self.activation_map:
            raise ValueError(f"Unsupported activation function: {self.activation}")
            

        # Multilayer Perceptron 
        self.mlp = nn.Sequential(
            nn.Linear(d_hidden, d_mlp),
            self.activation_map[self.activation](), 
            nn.Dropout(dropout),
            nn.Linear(d_mlp, d_hidden)
        )
        
    def forward(self, x): 
        # Pre-Norm Multi-Head Attention 
        norm_x = self.norm1(x) 
        attn_output = self.attention(norm_x)  
        x = x + self.dropout1(attn_output)
        
        # Post-Norm Feed Forward Network
        norm_x = self.norm2(x)  
        mlp_output = self.mlp(norm_x)
        x = x + self.dropout2(mlp_output)  
        return x

"""Transformer Encoder with DropPath - Stochastic Depth"""
# Used for Swin Transformer Experiments
class TransformerEncoder_DropPath(nn.Module): 
    def __init__(self, args, d_hidden, d_mlp, num_heads, dropout, attention_dropout, drop_path):
        super(TransformerEncoder_DropPath, self).__init__()
        self.args = args 

        self.d_hidden = d_hidden 
        self.d_mlp = d_mlp
        self.num_heads = num_heads
        self.dropout = dropout
        self.attention_dropout = attention_dropout

        self.attention = MultiHeadAttention(d_hidden, num_heads, attention_dropout)

        self.norm1 = nn.Identity()
        self.norm2 = nn.Identity()
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

        # Activation Selection
        self.activation = args.activation

        # Activation function mapping
        self.activation_map = {
            "relu": lambda: nn.ReLU(inplace=args.inplace), 
            "silu": lambda: nn.SiLU(inplace=args.inplace), 
            "gelu": lambda: nn.GELU(), 
            "sigmoid": lambda: nn.Sigmoid(), 

            # Previous Activation Generation
            "gelu_s": lambda: GELU_s(sigma=args.sigma, inplace=args.inplace), 
            "silu_s": lambda: SiLU_s(sigma=args.sigma, inplace=args.inplace), 
            "zilu_old": lambda: ZiLU_Old(sigma=args.sigma, inplace=args.inplace), 

            # Current Activation Generation 
            "arctan": lambda: ArcTan(sigma=args.sigma), 
            "arctan_approx": lambda: ArcTan_Approx(sigma=args.sigma), 
            "zilu": lambda: ZiLU(sigma=args.sigma), 
            "zilu_approx": lambda: ZiLU_Approx(sigma=args.sigma), 
            "squareplus": lambda: SquarePlus(beta=4),

            # Other Activations
            "leaky_relu": lambda: nn.LeakyReLU(inplace=args.inplace), 
            "prelu": lambda: nn.PReLU(), 
            "elu": lambda: nn.ELU(inplace=args.inplace), 
            "hardshrink": lambda: nn.Hardshrink(), 
            "softshrink": lambda: nn.Softshrink(), 
            "tanhshrink": lambda: nn.Tanhshrink(), 
            "softplus": lambda: nn.Softplus(),
            "softsign": lambda: nn.Softsign(), 
            "tanh": lambda: nn.Tanh(),
            "celu": lambda: nn.CELU(inplace=args.inplace),
            "mish": lambda: nn.Mish(inplace=args.inplace), 
            "hardswish": lambda: nn.Hardswish(inplace=args.inplace), 
            "hardsigmoid": lambda: nn.Hardsigmoid(inplace=args.inplace),
            "selu": lambda: nn.SELU(inplace=args.inplace),
            "hardtanh": lambda: nn.Hardtanh(inplace=args.inplace), 
            "identity": lambda: nn.Identity()
        }

        if self.activation not in self.activation_map:
            raise ValueError(f"Unsupported activation function: {self.activation}")
            

        # Multilayer Perceptron 
        self.mlp = nn.Sequential(
            nn.Linear(d_hidden, d_mlp),
            self.activation_map[self.activation](), 
            nn.Dropout(dropout),
            nn.Linear(d_mlp, d_hidden)
        )
        
    def forward(self, x): 
        # Pre-Norm Multi-Head Attention 
        norm_x = self.norm1(x) 
        attn_output = self.attention(norm_x)  
        x = x + self.drop_path(attn_output)
        
        # Post-Norm Feed Forward Network
        norm_x = self.norm2(x)  
        mlp_output = self.mlp(norm_x)
        x = x + self.drop_path(mlp_output)  
        return x

"""Also can use timm's DropPath Implementation"""
# from timm.models.layers import DropPath
class DropPath(nn.Module):
    """Drop paths (Stochastic Depth) per sample  (when applied in main path of residual blocks)."""
    def __init__(self, drop_prob: float = 0.0):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob
        
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()  # binarize
        output = x.div(keep_prob) * random_tensor
        return output

vit_nolayernorm = ViT(vit_args).to(vit_args.device)

In [70]:
vit_nolayernorm_pre_activations = {}

for name, module in vit_nolayernorm.named_modules():
    if isinstance(module, ZiLU):
        module.register_forward_hook(make_hook(name, vit_nolayernorm_pre_activations))

# vit_nolayernorm.eval()
for images, labels in cifar100_vit_dataset.test_loader:
    images, labels = images.to(vit_nolayernorm.device), labels.to(vit_nolayernorm.device)
    outputs = vit_nolayernorm(images)
    break


hill_estimations = 0
for name, acts in vit_nolayernorm_pre_activations.items():
    tail_index = hill_estimator(acts.numpy(), k=100)
    print(f"Tail index for {name}: {tail_index}")
    hill_estimations += tail_index 

print()
print(f"Average Tail Index: {hill_estimations/len(vit_nolayernorm_pre_activations)}")


    


Tail index for transformer_encoder.0.mlp.1: 36.524257350387664
Tail index for transformer_encoder.1.mlp.1: 25.985970327833183
Tail index for transformer_encoder.2.mlp.1: 20.790659969856225
Tail index for transformer_encoder.3.mlp.1: 26.252797998672808
Tail index for transformer_encoder.4.mlp.1: 24.439838594548924
Tail index for transformer_encoder.5.mlp.1: 18.769752905207536
Tail index for transformer_encoder.6.mlp.1: 24.990008296033754
Tail index for transformer_encoder.7.mlp.1: 23.17220774140939
Tail index for transformer_encoder.8.mlp.1: 15.560456748293166
Tail index for transformer_encoder.9.mlp.1: 25.87768761552418
Tail index for transformer_encoder.10.mlp.1: 23.295849395973633
Tail index for transformer_encoder.11.mlp.1: 22.95096499241097

Average Tail Index: 24.050870994679283


## III. GPT2

In [3]:
# Dataset 
from dataset import WikiText103 

    
gpt_args = SimpleNamespace(
    vocab_size=50257,
    max_seq_length=1024,
    embedding_dim=768,
    num_attention_heads=12,
    num_layers=12,
    dropout=0.1,
    activation='zilu', 
    sigma=1.0,
    dataset='wikitext103',
    persistent_workers=False,
    prefetch_factor=None,
    pin_memory=True,
    data_path='/mnt/research/j.farias/mkang2/Datasets',
    batch_size=16,
    num_epochs=10,
    use_amp=True,
    num_workers=4,
    device='cuda',

)

wiki_dataset = WikiText103(gpt_args)

Loading cached WikiText from /mnt/research/j.farias/mkang2/Datasets/wikitext103_cache_1024


/mnt/local/python3.11.8/lib/python3.11/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


#### i. GPT with LayerNorm

In [72]:
# Layernorm GPT2
from Models.activation import (GELU_s, SiLU_s, ZiLU_Old, ArcTan,
                               ArcTan_Approx, ZiLU, ZiLU_Approx, SquarePlus)



class GPT2(nn.Module):
    def __init__(self, 
                 args, 
                 vocab_size=50257, 
                 max_seq_length=1024,
                 embedding_dim=768,
                 num_attention_heads=12,
                 num_layers=12,
                 dropout=0.1,
                 device='cuda'):

        super(GPT2, self).__init__()

        self.args = args 

        self.vocab_size = vocab_size
        self.max_seq_length = max_seq_length
        self.embedding_dim = embedding_dim
        self.num_attention_heads = num_attention_heads
        self.num_layers = num_layers
        self.dropout = dropout
        
        self.device = device

        # Dropout 
        self.dropout = nn.Dropout(self.dropout)
        
        # Embeddings 
        self.token_embeddings = nn.Embedding(vocab_size, embedding_dim)     
        self.position_embeddings = nn.Embedding(max_seq_length, embedding_dim)
        
        # Transformer Blocks    
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(args, embedding_dim, num_attention_heads, embedding_dim * 4, max_seq_length, dropout)
            for _ in range(num_layers)
        ])

        # Final Layer Norm 
        self.layer_norm = nn.LayerNorm(embedding_dim)

        # Linear output layer 
        self.lm_head = nn.Linear(embedding_dim, vocab_size, bias=False)
        self.lm_head.weight = self.token_embeddings.weight
        
        # Initialize Weights 
        self.apply(self._init_weights)

        # Scaled Initialization for Residual Layers 
        for pn, p in self.named_parameters():
            if p.dim() > 1:
                if 'fc2' in pn or 'w_o' in pn:
                    p.data.normal_(mean=0.0, std=0.02 / math.sqrt(2 * num_layers))

        self.name = f"GPT2_{num_layers}L_{num_attention_heads}H_{embedding_dim}D_{args.activation}"
        self.to(device)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)

    def forward(self, input_ids, target=None):
        batch_size, seq_length = input_ids.size()

        # Create position ids 
        position_ids = torch.arange(0, seq_length, dtype=torch.long, device=self.device)

        # Embeddings 
        token_embeds = self.token_embeddings(input_ids)
        position_embeds = self.position_embeddings(position_ids)
        x = token_embeds + position_embeds

        x = self.dropout(x)

        # Transformer Blocks 
        for layer in self.transformer_blocks:
            x = layer(x)

        # Final Layer Norm 
        x = self.layer_norm(x)

        if target is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), target.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None 

        return logits, loss
    
    def parameter_count(self, non_embeddings=True): 
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        
        return total_params, trainable_params
    
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, d_embeddings, num_heads, max_seq_length, dropout):
        super(CausalMultiHeadAttention, self).__init__()

        assert d_embeddings % num_heads == 0, "Match Embeddings with Number of Heads"

        self.d_embedding = d_embeddings
        self.num_heads = num_heads
        self.max_seq_length = max_seq_length
        self.d_heads = d_embeddings // num_heads


        self.w_k = nn.Linear(d_embeddings, d_embeddings)
        self.w_q = nn.Linear(d_embeddings, d_embeddings)
        self.w_v = nn.Linear(d_embeddings, d_embeddings)
        self.w_o = nn.Linear(d_embeddings, d_embeddings)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_length, max_seq_length)).view(1, 1, max_seq_length, max_seq_length)
        self.register_buffer('causal_mask', causal_mask)

    def split_heads(self, x):
        batch_size, seq_length, d_embeddings = x.shape 
        return x.view(batch_size, seq_length, self.num_heads, self.d_heads).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, num_heads, seq_length, d_heads = x.shape 
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, num_heads * d_heads)        
        
    def forward(self, x):
        k = self.split_heads(self.w_k(x))
        q = self.split_heads(self.w_q(x))
        v = self.split_heads(self.w_v(x))        

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * (1.0 / math.sqrt(v.size(-1)))

        seq_length = attn_scores.size(-2)
        mask_slice = self.causal_mask[:, :, :seq_length, :seq_length]
        attn_scores = attn_scores.masked_fill(mask_slice == 0, float('-inf'))

        # Softmax and weighting 
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.attn_dropout(attn_probs)
        attn_output = torch.matmul(attn_probs, v)

        attn_output = self.combine_heads(attn_output)
        attn_output = self.w_o(attn_output)
        attn_output = self.resid_dropout(attn_output)
        return attn_output

class MLP(nn.Module):
    def __init__(self, args, d_model, d_ff, dropout):
        super(MLP, self).__init__() 

         # Activation Selection
        self.activation = args.activation

        # Activation function mapping
        self.activation_map = {
            "relu": lambda: nn.ReLU(inplace=args.inplace), 
            "silu": lambda: nn.SiLU(inplace=args.inplace), 
            "gelu": lambda: nn.GELU(), 
            "sigmoid": lambda: nn.Sigmoid(), 

            # Previous Activation Generation
            "gelu_s": lambda: GELU_s(sigma=args.sigma, inplace=args.inplace), 
            "silu_s": lambda: SiLU_s(sigma=args.sigma, inplace=args.inplace), 
            "zilu_old": lambda: ZiLU_Old(sigma=args.sigma, inplace=args.inplace), 

            # Current Activation Generation 
            "arctan": lambda: ArcTan(sigma=args.sigma), 
            "arctan_approx": lambda: ArcTan_Approx(sigma=args.sigma), 
            "zilu": lambda: ZiLU(sigma=args.sigma), 
            "zilu_approx": lambda: ZiLU_Approx(sigma=args.sigma), 
            "squareplus": lambda: SquarePlus(beta=4),

            # Other Activations
            "leaky_relu": lambda: nn.LeakyReLU(inplace=args.inplace), 
            "prelu": lambda: nn.PReLU(), 
            "elu": lambda: nn.ELU(inplace=args.inplace), 
            "hardshrink": lambda: nn.Hardshrink(), 
            "softshrink": lambda: nn.Softshrink(), 
            "tanhshrink": lambda: nn.Tanhshrink(), 
            "softplus": lambda: nn.Softplus(),
            "softsign": lambda: nn.Softsign(), 
            "tanh": lambda: nn.Tanh(),
            "celu": lambda: nn.CELU(inplace=args.inplace),
            "mish": lambda: nn.Mish(inplace=args.inplace), 
            "hardswish": lambda: nn.Hardswish(inplace=args.inplace), 
            "hardsigmoid": lambda: nn.Hardsigmoid(inplace=args.inplace),
            "selu": lambda: nn.SELU(inplace=args.inplace),
            "hardtanh": lambda: nn.Hardtanh(inplace=args.inplace), 
            "identity": lambda: nn.Identity()
        }

        if self.activation not in self.activation_map:
            raise ValueError(f"Unsupported activation function: {self.activation}")
            
        
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.activation_function = self.activation_map[self.activation]()

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation_function(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

class TransformerBlock(nn.Module):
    def __init__(self, args, d_model, num_heads, d_ff, max_seq_length, dropout):
        super(TransformerBlock, self).__init__()
        self.attention = CausalMultiHeadAttention(d_model, num_heads, max_seq_length, dropout)
        self.mlp = MLP(args, d_model, d_ff, dropout)
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.layer_norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # Pre-Norm Multi-Head Attention 
        norm_x = self.layer_norm1(x)
        attn_output = self.attention(norm_x)
        x = x + attn_output

        # Post-Norm MLP
        norm_x = self.layer_norm2(x)
        mlp_output = self.mlp(norm_x)
        x = x + mlp_output
        return x


torch.cuda.empty_cache()

# Now instantiate GPT-2
gpt_layernorm = GPT2(gpt_args)

In [73]:
gpt_layernorm_pre_activations = {}

for name, module in gpt_layernorm.named_modules():
    if isinstance(module, ZiLU):
        module.register_forward_hook(make_hook(name, gpt_layernorm_pre_activations))

# gpt_layernorm.eval()
for batch in wiki_dataset.val_loader:
    tokens = batch['input_ids'].to(gpt_args.device)
    inputs = tokens[:, :-1].contiguous()
    targets = tokens[:, 1:].contiguous()

    logits, loss = gpt_layernorm(inputs, targets)
    break  # Just one batch for testing

hill_estimations = 0
for name, acts in gpt_layernorm_pre_activations.items():
    tail_index = hill_estimator(acts.numpy(), k=100)
    print(f"Tail index for {name}: {tail_index}")
    hill_estimations += tail_index 

print()
print(f"Average Tail Index: {hill_estimations/len(gpt_layernorm_pre_activations)}")




Tail index for transformer_blocks.0.mlp.activation_function: 26.57051127739529
Tail index for transformer_blocks.1.mlp.activation_function: 26.675042981293018
Tail index for transformer_blocks.2.mlp.activation_function: 27.98376445489523
Tail index for transformer_blocks.3.mlp.activation_function: 26.47634203515341
Tail index for transformer_blocks.4.mlp.activation_function: 23.43010931806118
Tail index for transformer_blocks.5.mlp.activation_function: 21.65462824062754
Tail index for transformer_blocks.6.mlp.activation_function: 24.348625685299698
Tail index for transformer_blocks.7.mlp.activation_function: 26.39676369176227
Tail index for transformer_blocks.8.mlp.activation_function: 29.204913812755898
Tail index for transformer_blocks.9.mlp.activation_function: 29.34066017856492
Tail index for transformer_blocks.10.mlp.activation_function: 27.79392530059198
Tail index for transformer_blocks.11.mlp.activation_function: 23.726694624499945

Average Tail Index: 26.1334984667417


#### ii. GPT2 no LayerNorm

In [4]:
# Layernorm GPT2
from Models.activation import (GELU_s, SiLU_s, ZiLU_Old, ArcTan,
                               ArcTan_Approx, ZiLU, ZiLU_Approx, SquarePlus)



class GPT2(nn.Module):
    def __init__(self, 
                 args, 
                 vocab_size=50257, 
                 max_seq_length=1024,
                 embedding_dim=768,
                 num_attention_heads=12,
                 num_layers=12,
                 dropout=0.1,
                 device='cuda'):

        super(GPT2, self).__init__()

        self.args = args 

        self.vocab_size = vocab_size
        self.max_seq_length = max_seq_length
        self.embedding_dim = embedding_dim
        self.num_attention_heads = num_attention_heads
        self.num_layers = num_layers
        self.dropout = dropout
        
        self.device = device

        # Dropout 
        self.dropout = nn.Dropout(self.dropout)
        
        # Embeddings 
        self.token_embeddings = nn.Embedding(vocab_size, embedding_dim)     
        self.position_embeddings = nn.Embedding(max_seq_length, embedding_dim)
        
        # Transformer Blocks    
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(args, embedding_dim, num_attention_heads, embedding_dim * 4, max_seq_length, dropout)
            for _ in range(num_layers)
        ])

        # Final Layer Norm 
        self.layer_norm = nn.Identity()  # No LayerNorm

        # Linear output layer 
        self.lm_head = nn.Linear(embedding_dim, vocab_size, bias=False)
        self.lm_head.weight = self.token_embeddings.weight
        
        # Initialize Weights 
        self.apply(self._init_weights)

        # Scaled Initialization for Residual Layers 
        for pn, p in self.named_parameters():
            if p.dim() > 1:
                if 'fc2' in pn or 'w_o' in pn:
                    p.data.normal_(mean=0.0, std=0.02 / math.sqrt(2 * num_layers))

        self.name = f"GPT2_{num_layers}L_{num_attention_heads}H_{embedding_dim}D_{args.activation}"
        self.to(device)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)

    def forward(self, input_ids, target=None):
        batch_size, seq_length = input_ids.size()

        # Create position ids 
        position_ids = torch.arange(0, seq_length, dtype=torch.long, device=self.device)

        # Embeddings 
        token_embeds = self.token_embeddings(input_ids)
        position_embeds = self.position_embeddings(position_ids)
        x = token_embeds + position_embeds

        x = self.dropout(x)

        # Transformer Blocks 
        for layer in self.transformer_blocks:
            x = layer(x)

        # Final Layer Norm 
        x = self.layer_norm(x)

        if target is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), target.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None 

        return logits, loss
    
    def parameter_count(self, non_embeddings=True): 
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        
        return total_params, trainable_params
    
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, d_embeddings, num_heads, max_seq_length, dropout):
        super(CausalMultiHeadAttention, self).__init__()

        assert d_embeddings % num_heads == 0, "Match Embeddings with Number of Heads"

        self.d_embedding = d_embeddings
        self.num_heads = num_heads
        self.max_seq_length = max_seq_length
        self.d_heads = d_embeddings // num_heads


        self.w_k = nn.Linear(d_embeddings, d_embeddings)
        self.w_q = nn.Linear(d_embeddings, d_embeddings)
        self.w_v = nn.Linear(d_embeddings, d_embeddings)
        self.w_o = nn.Linear(d_embeddings, d_embeddings)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_length, max_seq_length)).view(1, 1, max_seq_length, max_seq_length)
        self.register_buffer('causal_mask', causal_mask)

    def split_heads(self, x):
        batch_size, seq_length, d_embeddings = x.shape 
        return x.view(batch_size, seq_length, self.num_heads, self.d_heads).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, num_heads, seq_length, d_heads = x.shape 
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, num_heads * d_heads)        
        
    def forward(self, x):
        k = self.split_heads(self.w_k(x))
        q = self.split_heads(self.w_q(x))
        v = self.split_heads(self.w_v(x))        

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * (1.0 / math.sqrt(v.size(-1)))

        seq_length = attn_scores.size(-2)
        mask_slice = self.causal_mask[:, :, :seq_length, :seq_length]
        attn_scores = attn_scores.masked_fill(mask_slice == 0, float('-inf'))

        # Softmax and weighting 
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.attn_dropout(attn_probs)
        attn_output = torch.matmul(attn_probs, v)

        attn_output = self.combine_heads(attn_output)
        attn_output = self.w_o(attn_output)
        attn_output = self.resid_dropout(attn_output)
        return attn_output

class MLP(nn.Module):
    def __init__(self, args, d_model, d_ff, dropout):
        super(MLP, self).__init__() 

         # Activation Selection
        self.activation = args.activation

        # Activation function mapping
        self.activation_map = {
            "relu": lambda: nn.ReLU(inplace=args.inplace), 
            "silu": lambda: nn.SiLU(inplace=args.inplace), 
            "gelu": lambda: nn.GELU(), 
            "sigmoid": lambda: nn.Sigmoid(), 

            # Previous Activation Generation
            "gelu_s": lambda: GELU_s(sigma=args.sigma, inplace=args.inplace), 
            "silu_s": lambda: SiLU_s(sigma=args.sigma, inplace=args.inplace), 
            "zilu_old": lambda: ZiLU_Old(sigma=args.sigma, inplace=args.inplace), 

            # Current Activation Generation 
            "arctan": lambda: ArcTan(sigma=args.sigma), 
            "arctan_approx": lambda: ArcTan_Approx(sigma=args.sigma), 
            "zilu": lambda: ZiLU(sigma=args.sigma), 
            "zilu_approx": lambda: ZiLU_Approx(sigma=args.sigma), 
            "squareplus": lambda: SquarePlus(beta=4),

            # Other Activations
            "leaky_relu": lambda: nn.LeakyReLU(inplace=args.inplace), 
            "prelu": lambda: nn.PReLU(), 
            "elu": lambda: nn.ELU(inplace=args.inplace), 
            "hardshrink": lambda: nn.Hardshrink(), 
            "softshrink": lambda: nn.Softshrink(), 
            "tanhshrink": lambda: nn.Tanhshrink(), 
            "softplus": lambda: nn.Softplus(),
            "softsign": lambda: nn.Softsign(), 
            "tanh": lambda: nn.Tanh(),
            "celu": lambda: nn.CELU(inplace=args.inplace),
            "mish": lambda: nn.Mish(inplace=args.inplace), 
            "hardswish": lambda: nn.Hardswish(inplace=args.inplace), 
            "hardsigmoid": lambda: nn.Hardsigmoid(inplace=args.inplace),
            "selu": lambda: nn.SELU(inplace=args.inplace),
            "hardtanh": lambda: nn.Hardtanh(inplace=args.inplace), 
            "identity": lambda: nn.Identity()
        }

        if self.activation not in self.activation_map:
            raise ValueError(f"Unsupported activation function: {self.activation}")
            
        
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.activation_function = self.activation_map[self.activation]()

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation_function(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

class TransformerBlock(nn.Module):
    def __init__(self, args, d_model, num_heads, d_ff, max_seq_length, dropout):
        super(TransformerBlock, self).__init__()
        self.attention = CausalMultiHeadAttention(d_model, num_heads, max_seq_length, dropout)
        self.mlp = MLP(args, d_model, d_ff, dropout)
        self.layer_norm1 = nn.Identity()  # No LayerNorm
        self.layer_norm2 = nn.Identity()  # No LayerNorm

    def forward(self, x):
        # Pre-Norm Multi-Head Attention 
        norm_x = self.layer_norm1(x)
        attn_output = self.attention(norm_x)
        x = x + attn_output

        # Post-Norm MLP
        norm_x = self.layer_norm2(x)
        mlp_output = self.mlp(norm_x)
        x = x + mlp_output
        return x

import gc
import torch

# # Delete all previous models and activation dicts
# # del resnet_batchnorm
# del resnet_nobatchnorm
# del vit_layernorm
# del vit_nolayernorm
# del resnet_batchnorm_pre_activations
# del resnet_nobatchnorm_pre_activations
# del vit_layernorm_pre_activations
# del vit_nolayernorm_pre_activations
# del gpt_layernorm_pre_activations 
# del gpt_layernorm

# gc.collect()
torch.cuda.empty_cache()

# Now instantiate GPT-2
gpt_nolayernorm = GPT2(gpt_args)

In [5]:
gpt_nolayernorm_pre_activations = {}

for name, module in gpt_nolayernorm.named_modules():
    if isinstance(module, ZiLU):
        module.register_forward_hook(make_hook(name, gpt_nolayernorm_pre_activations))

# gpt_nolayernorm.eval()
for batch in wiki_dataset.val_loader:
    tokens = batch['input_ids'].to(gpt_args.device)
    inputs = tokens[:, :-1].contiguous()
    targets = tokens[:, 1:].contiguous()

    logits, loss = gpt_nolayernorm(inputs, targets)
    break  # Just one batch for testing

hill_estimations = 0
for name, acts in gpt_nolayernorm_pre_activations.items():
    tail_index = hill_estimator(acts.numpy(), k=100)
    print(f"Tail index for {name}: {tail_index}")
    hill_estimations += tail_index 

print()
print(f"Average Tail Index: {hill_estimations/len(gpt_nolayernorm_pre_activations)}")




Tail index for transformer_blocks.0.mlp.activation_function: 25.597815811370587
Tail index for transformer_blocks.1.mlp.activation_function: 26.89118387004947
Tail index for transformer_blocks.2.mlp.activation_function: 27.07372685893429
Tail index for transformer_blocks.3.mlp.activation_function: 22.564219980142344
Tail index for transformer_blocks.4.mlp.activation_function: 24.218566129215912
Tail index for transformer_blocks.5.mlp.activation_function: 23.129714322961156
Tail index for transformer_blocks.6.mlp.activation_function: 31.54035809172255
Tail index for transformer_blocks.7.mlp.activation_function: 23.62734457922434
Tail index for transformer_blocks.8.mlp.activation_function: 25.78610726304614
Tail index for transformer_blocks.9.mlp.activation_function: 24.159033948297655
Tail index for transformer_blocks.10.mlp.activation_function: 23.651745230182357
Tail index for transformer_blocks.11.mlp.activation_function: 23.234178112021393

Average Tail Index: 25.122832849764013
